In [1]:
#packages
import requests
import pandas as pd
import json
from glob import glob
from pathlib import Path
import csv
import numpy as np
import matplotlib.pyplot as plt
from censusdis import data
from censusdis.datasets import ACS5
from scipy.spatial import cKDTree
from shapely.geometry import Point
import geopandas as gpd

# Socioeconomic dataset
One row per tract.

Returns: tract_socioeconomic


In [2]:
vars = [
    # Population
    "B01003_001E",    # total population

    # Income & Poverty
    "B19013_001E",    # median household income
    "C17002_002E",    # below 50% poverty (numerator for severe poverty)

    # Housing & Vehicles (numerators + denominators)
    "B25044_001E",    # households for vehicle availability (denominator)
    "B25044_003E",    # households with NO vehicle (numerator)

    # Race / Ethnicity (counts)
    "B02001_002E", "B02001_003E", "B02001_004E",
    "B02001_005E", "B02001_006E", "B02001_007E", "B02001_008E",

    # Housing Cost Burden (rent % of income)
    "B25070_001E", "B25070_007E", "B25070_008E", "B25070_009E", "B25070_010E",

    # Education
    "B15003_022E",   # bachelor's degree (if we want bachelors+, consider adding 023-025)

    # Disability (numerator)
    "B18135_002E",   # population WITH a disability (numerator)
    # NOTE: also add B18135_001E (denominator) below

    # Internet Access (numerator)
    "B28002_002E",   # households with broadband subscription (numerator)
    # NOTE: also add B28002_001E (denominator) below

    # Age 65+ (WE ADD the full set of bins to sum up correctly)
    # Male 65-66 .. 85+
    "B01001_020E","B01001_021E","B01001_022E","B01001_023E","B01001_024E","B01001_025E",
    # Female 65-66 .. 85+
    "B01001_044E","B01001_045E","B01001_046E","B01001_047E","B01001_048E","B01001_049E",

    # Housing Tenure (optional)
    "B25003_003E",   # renter-occupied units (numerator)
    "B25003_001E",   # total housing units for tenure table (denominator)
    
    # Disability and internet DENOMINATORS (needed to compute rates correctly)
    "B18135_001E",   # total civilian noninstitutionalized population (disability denominator)
    "B28002_001E"    # total households for internet table (internet denominator)
]

#Pulling this data for all census tracts in NC

In [3]:
dataset = "acs/acs5"
vintage = 2022        #2024 comes out on Dec 11
state_fips = "37"     # North Carolina as string

df = data.download(
    dataset=dataset,
    vintage=vintage,
    download_variables=vars,
    # Geographic filters (must be strings)
    state=state_fips,
    county="*",        # all counties
    tract="*"          # all tracts within each county
)

#derived indicators

#Getting percentages, replace any with NA denominator with 0
# safe pct helper (keeps your original fill-0 behavior)
def pct(num, den):
    den_safe = den.replace(0, np.nan)
    result = num / den_safe
    return result.fillna(0)

# Severe poverty
df["pct_below_50pct_poverty"] = pct(df["C17002_002E"], df["B01003_001E"])

# No vehicle
df["pct_no_vehicle"] = pct(df["B25044_003E"], df["B25044_001E"])

# Renters cost burden (sum of 30%+ buckets / total renters)
df["pct_renters_cost_burdened"] = pct(
    df["B25070_007E"] + df["B25070_008E"] + df["B25070_009E"] + df["B25070_010E"],
    df["B25070_001E"]
)

# Percent 65+ : sum the male+female 65+ bins, divide by total population (B01003_001E)
age_cols_65 = [
    "B01001_020E","B01001_021E","B01001_022E","B01001_023E","B01001_024E","B01001_025E",
    "B01001_044E","B01001_045E","B01001_046E","B01001_047E","B01001_048E","B01001_049E"
]
df["population_over65"] = df[age_cols_65].sum(axis=1)
df["pct_over65"] = pct(df["population_over65"], df["B01003_001E"])

# Disability: use ACS numerator (people with disability) / ACS denominator (civilian noninstitutionalized)
# numerator: B18135_002E ; denominator: B18135_001E
df["pct_with_disability"] = pct(df["B18135_002E"], df["B18135_001E"])

# Optional broadband if we want (uncomment if downloaded B28002_001E)
# df["pct_broadband"] = pct(df["B28002_002E"], df["B28002_001E"])

# Percent nonwhite: 1 - (white / total_race_counts)
race_cols = ["B02001_002E","B02001_003E","B02001_004E","B02001_005E","B02001_006E","B02001_007E","B02001_008E"]
race_total = df[race_cols].sum(axis=1)
df["pct_nonwhite"] = 1 - pct(df["B02001_002E"], race_total)
#df["pct_renters"] = pct(df["B25003_003E"], df["B25003_001E"])

#save dataframe
#df.to_csv("./data_processed/nc_census_tracts_acs5.csv", index=False)

'''
#downloading tract boundaries for mapping
df_geo = data.download(dataset=dataset, vintage=year,
                       variables=vars, geography="tract",
                       state=state, with_geometry=True)
df_geo.to_file("nc_census_tracts_acs5.geojson", driver="GeoJSON")
'''

'\n#downloading tract boundaries for mapping\ndf_geo = data.download(dataset=dataset, vintage=year,\n                       variables=vars, geography="tract",\n                       state=state, with_geometry=True)\ndf_geo.to_file("nc_census_tracts_acs5.geojson", driver="GeoJSON")\n'

In [11]:
df['STATE']  = df['STATE'].astype(str).str.zfill(2)
df['COUNTY'] = df['COUNTY'].astype(str).str.zfill(3)
df['TRACT']  = df['TRACT'].astype(str).str.zfill(6)

# 2. Build full GEOID-like unique identifier
df['TRACT'] = df['STATE'] + df['COUNTY'] + df['TRACT']

duplicate_counts = df['TRACT'].duplicated().sum()
print("Number of duplicate TRACT values:", duplicate_counts)

Number of duplicate TRACT values: 0


In [12]:
rename_dict = {
    # Population
    "B01003_001E": "total_population",

    # Income & Poverty
    "B19013_001E": "median_household_income",
    "C17002_002E": "population_below_50pct_poverty",

    # Housing & Vehicles
    "B25044_001E": "total_households_vehicle_data",
    "B25044_003E": "households_no_vehicle",

    # Race / Ethnicity counts
    "B02001_002E": "white_alone",
    "B02001_003E": "black_alone",
    "B02001_004E": "american_indian_alaska_native",
    "B02001_005E": "asian_alone",
    "B02001_006E": "native_hawaiian_pacific_islander",
    "B02001_007E": "some_other_race",
    "B02001_008E": "two_or_more_races",

    # Housing Cost Burden (Gross Rent as % of Income)
    "B25070_001E": "total_renters",
    "B25070_007E": "rent_30_to_34_pct_income",
    "B25070_008E": "rent_35_to_39_pct_income",
    "B25070_009E": "rent_40_to_49_pct_income",
    "B25070_010E": "rent_50_plus_pct_income",

    # Education (bachelor’s only, unless adding more later)
    "B15003_022E": "bachelors_degree",

    # Disability (denominator + numerator)
    "B18135_001E": "population_civilian_noninstitutionalized",
    "B18135_002E": "population_with_disability",

    # Internet Access
    "B28002_001E": "total_households_in_internet_table",
    "B28002_002E": "households_with_broadband",

    # Age 65+ bins (male + female)
    "B01001_020E": "male_65_66",
    "B01001_021E": "male_67_69",
    "B01001_022E": "male_70_74",
    "B01001_023E": "male_75_79",
    "B01001_024E": "male_80_84",
    "B01001_025E": "male_85_plus",

    "B01001_044E": "female_65_66",
    "B01001_045E": "female_67_69",
    "B01001_046E": "female_70_74",
    "B01001_047E": "female_75_79",
    "B01001_048E": "female_80_84",
    "B01001_049E": "female_85_plus",

    # Housing Tenure
    "B25003_001E": "total_housing_units",
    "B25003_003E": "renter_occupied_housing_units",
}


df = df.rename(columns=rename_dict)


In [ ]:
from sklearn.preprocessing import StandardScaler

#Just wanted to save the unscaled version as well
tract_socioeconomic_unscaled = df

# Scale all numeric columns EXCEPT TRACT, STATE, COUNTY
cols_to_scale = df.columns.difference(['TRACT', 'STATE', 'COUNTY'])

scaler = StandardScaler()
scaled_values = scaler.fit_transform(df[cols_to_scale])

df_scaled = pd.DataFrame(
    scaled_values,
    columns=cols_to_scale,
    index=df.index
)

# Add back TRACT id
df_scaled['TRACT'] = df['TRACT']

# Drop STATE and COUNTY from df_scaled (now correctly)
df_scaled = df_scaled.drop(columns=['STATE', 'COUNTY'], errors='ignore')

# Final PCA-ready socioeconomic dataframe (data has been scaled)
tract_socioeconomic = df_scaled

# Outage-level dataset

Returns: outage_events

In [2]:
import re

outages_raw = pd.read_csv('/Users/JChuang/Documents/3S MP/MP/data_raw/outage_tracker.csv')

#Filter out North Carolina
outages_NC = outages_raw[outages_raw['state'] == "North Carolina"]
#Turn outage_start_estimate and outage_end_estimate to UTC datetime objects
outages_NC['outage_start_estimate'] = pd.to_datetime(outages_NC['outage_start_estimate'], utc=True).dt.tz_localize(None).dt.floor('S')
outages_NC['outage_end_estimate'] = pd.to_datetime(outages_NC['outage_end_estimate'], utc=True).dt.tz_localize(None).dt.floor('S')

#Turn fix_duration_estimate into hours
def duration_to_hours(duration_str):
    if pd.isna(duration_str):
        return None
    match = re.match(r'(?:(\d+)d )?(?:(\d+)h )?(?:(\d+)m )?(?:(\d+)s)?', duration_str)
    if not match:
        return None
    days = int(match.group(1)) if match.group(1) else 0
    hours = int(match.group(2)) if match.group(2) else 0
    minutes = int(match.group(3)) if match.group(3) else 0
    seconds = int(match.group(4)) if match.group(4) else 0
    total_hours = days * 24 + hours + minutes / 60 + seconds / 3600
    return total_hours

outages_NC['fix_duration_hours'] = outages_NC['fix_duration_estimate'].apply(duration_to_hours)

outages_NC = outages_NC.sort_values("outage_start_estimate").reset_index(drop=True)


/var/folders/x4/39bns7ns63g9446c_4gzy_tr0000gn/T/ipykernel_61423/705568491.py:8: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  outages_NC['outage_start_estimate'] = pd.to_datetime(outages_NC['outage_start_estimate'], utc=True).dt.tz_localize(None).dt.floor('S')
/var/folders/x4/39bns7ns63g9446c_4gzy_tr0000gn/T/ipykernel_61423/705568491.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  outages_NC['outage_start_estimate'] = pd.to_datetime(outages_NC['outage_start_estimate'], utc=True).dt.tz_localize(None).dt.floor('S')
/var/folders/x4/39bns7ns63g9446c_4gzy_tr0000gn/T/ipykernel_61423/705568491.py:9: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.

In [3]:
#I am identifying the counties whose outage data during hurricane helene that I will remove
#western NC county names identified as mountain counties in NC

# ---- CONFIG ----
#WESTERN_COUNTIES = {
#    'Alleghany','Ashe','Avery','Buncombe','Burke','Caldwell','Cherokee','Clay',
#    'Graham','Haywood','Henderson','Jackson','Macon','Madison','McDowell',
#    'Mitchell','Polk','Rutherford','Swain','Transylvania','Watauga','Wilkes','Yancey'
#}

# Western NC counties (Mountain region) — county FIPS codes (3-digit)
# State FIPS for NC = 37
WESTERN_NC_COUNTYFPS = {
    '001','005','009','011','021','027','039','043','045','087','089',
    '099','113','115','121','149','161','175','189','199','221','193','197'
}

# Ensure block_fips is string
outages_NC['block_fips'] = outages_NC['block_fips'].astype(str)

# Extract TRACT (first 11 chars of block GEOID)
outages_NC['TRACT'] = outages_NC['block_fips'].str[0:11]

# Extract county FIPS from tract GEOID (state(2) + county(3))
outages_NC['countyfp'] = outages_NC['TRACT'].str[2:5]

# Ensure outage start/end are datetimes in UTC
outages_NC['outage_start_estimate'] = pd.to_datetime(
    outages_NC['outage_start_estimate'], utc=True
)
outages_NC['outage_end_estimate'] = pd.to_datetime(
    outages_NC['outage_end_estimate'], utc=True
)

# Define Helene UTC window
start_helene = pd.Timestamp("2024-09-26 00:00", tz="UTC")
end_helene   = pd.Timestamp("2024-09-30 23:59", tz="UTC")

# Time overlap flag
outages_NC['overlaps_helene'] = (
    (outages_NC['outage_start_estimate'] <= end_helene) &
    (outages_NC['outage_end_estimate'] >= start_helene)
)

# Western NC flag (county-based)
outages_NC['is_western_nc'] = outages_NC['countyfp'].isin(WESTERN_NC_COUNTYFPS)

# Final drop rule: overlap in time AND western NC
outages_NC['drop_for_helene'] = (
    outages_NC['overlaps_helene'] & outages_NC['is_western_nc']
)

# Quick diagnostics
print(f"Total outages: {len(outages_NC):,}")
print(f"Overlapping Helene (anywhere in NC): {outages_NC['overlaps_helene'].sum():,}")
print(f"Dropped (Helene overlap + western NC): {outages_NC['drop_for_helene'].sum():,}")

# Final cleaned dataframe
outage_events = outages_NC.loc[~outages_NC['drop_for_helene']].copy()

print(f"Final outage_events length: {len(outage_events):,}")

Total outages: 364,334
Overlapping Helene (anywhere in NC): 11,125
Dropped (Helene overlap + western NC): 7,019
Final outage_events length: 357,315


In [4]:
outage_events.columns

Index(['outage_identifer', 'device_lat', 'device_lon', 'block_fips',
       'convex_hull', 'jurisdiction', 'origin', 'state', 'county', 'affected',
       'cause', 'outage_start_estimate', 'outage_end_estimate',
       'fix_duration_estimate', 'outage_restored', 'fix_duration_hours',
       'TRACT', 'countyfp', 'overlaps_helene', 'is_western_nc',
       'drop_for_helene'],
      dtype='object')

# Tract-level outage dataset

Returns: tract_outages


In [5]:
#create customer minutes out
outage_events['customer_minutes_out'] = outage_events['fix_duration_hours'] * outage_events['affected'] * 60
#Add seasonal categories
outage_events['month'] = outage_events['outage_start_estimate'].dt.month
outage_events['is_summer'] = outage_events['month'].isin([6, 7, 8])
outage_events['is_winter'] = outage_events['month'].isin([12, 1, 2])


In [6]:
tract_outage_stats = outage_events.groupby("TRACT").agg(
    total_outages = ("outage_identifer", "count"),

    # Duration metrics
    mean_duration_hours = ("fix_duration_hours", "mean"),
    median_duration_hours = ("fix_duration_hours", "median"),
    max_duration_hours = ("fix_duration_hours", "max"),
    total_duration_hours = ("fix_duration_hours", "sum"),

    # Customers affected
    mean_customers_affected = ("affected", "mean"),
    max_customers_affected = ("affected", "max"),
    total_customers_affected = ("affected", "sum"),

    # CMO
    total_customer_minutes_out = ("customer_minutes_out", "sum"),

    # Seasonal
    pct_outages_in_summer = ("is_summer", "mean"),
    pct_outages_in_winter = ("is_winter", "mean"),
).reset_index()


In [30]:
tract_outages = tract_outage_stats

# Tract-level weather summary dataset

One row per tract.

Returns: tract_weather


In [21]:
#create a mapping: which tract gets which weather station name
data_folder = Path('/Users/JChuang/Documents/3S MP/MP/econet_data')

#Creating an empty dictionary to hold DataFrames for each station
station_dfs = {}

#Loop through all CSV files in the folder
for file in data_folder.glob("hourly_data_*.csv"):
    #Extract station name from filename
    station_name = file.stem.split("_")[-1]

    #Read CSV into DataFrame
    df = pd.read_csv(file)

    #Add to dictionary, appending if station already exists
    if station_name in station_dfs:
        station_dfs[station_name] = pd.concat([station_dfs[station_name], df], ignore_index = True)
    else:
        station_dfs[station_name] = df


#one combined DataFrame with a new column for station name
combined_df = pd.concat(
    [df.assign(station=station) for station, df in station_dfs.items()],
    ignore_index=True
)


In [ ]:
# Load NC tract shapefile from TIGER (2020 tracts for NC: state FIPS 37)
tiger_url = "https://www2.census.gov/geo/tiger/TIGER2020/TRACT/tl_2020_37_tract.zip"
gdf_tracts = gpd.read_file(tiger_url)

# Keep only necessary columns
gdf_tracts = gdf_tracts[['GEOID', 'geometry']].rename(columns={'GEOID': 'TRACT'})

# Compute centroids in a projected CRS for accurate distances
# Project to Web Mercator (meters) for distance calculations
gdf_tracts_proj = gdf_tracts.to_crs(epsg=3857)

# Compute centroids in projected CRS
gdf_tracts_proj['centroid_geom'] = gdf_tracts_proj.geometry.centroid

# Make a GeoDataFrame of centroids and transform back to EPSG:4326 (lon/lat) for readability
gdf_centroids = gpd.GeoDataFrame(
    gdf_tracts_proj[['TRACT']].copy(),
    geometry=gdf_tracts_proj['centroid_geom'],
    crs="EPSG:3857"
).to_crs(epsg=4326)

# Extract lon/lat
gdf_centroids['tract_lon'] = gdf_centroids.geometry.x
gdf_centroids['tract_lat'] = gdf_centroids.geometry.y

# Prepare weather station table (unique stations with lat/lon)
# If your combined_df has different column names, change them here
weather_stations = combined_df[['location_id', 'latitude_degrees_north', 'longitude_degrees_east']].drop_duplicates().rename(
    columns={'latitude_degrees_north': 'station_lat', 'longitude_degrees_east': 'station_lon', 'location_id':'station'}
).reset_index(drop=True)

# Convert stations to GeoDataFrame and project to same projected CRS (3857) for KDTree
gdf_stations = gpd.GeoDataFrame(
    weather_stations,
    geometry=gpd.points_from_xy(weather_stations['station_lon'], weather_stations['station_lat']),
    crs="EPSG:4326"
).to_crs(epsg=3857)

# Project centroids to 3857 for distance calculations
gdf_centroids_proj = gdf_centroids.to_crs(epsg=3857)

# Build KDTree on station coordinates (projected) and query nearest neighbor 
# Coordinates in meters
station_coords = np.vstack([gdf_stations.geometry.x.values, gdf_stations.geometry.y.values]).T
tract_coords = np.vstack([gdf_centroids_proj.geometry.x.values, gdf_centroids_proj.geometry.y.values]).T

tree = cKDTree(station_coords)
distances, idxs = tree.query(tract_coords, k=1)

# Assign nearest station name and distance (meters)
gdf_centroids['nearest_station'] = gdf_stations.iloc[idxs].station.values
gdf_centroids['distance_to_station_m'] = distances

# Save mapping table and (optionally) merge to your outages dataframe 
tract_station_map = gdf_centroids[['TRACT', 'tract_lat', 'tract_lon', 'nearest_station', 'distance_to_station_m']].copy()
tract_station_map['TRACT'] = tract_station_map['TRACT'].astype(str)

#Take out tract and county from block FIPS
tract_station_map['TRACT'] = tract_station_map['TRACT'].str[0:11]

Hurricane Helene impacted the Southeast U.S. roughly:

UTC window to remove:

Start: 2024-09-26 00:00 UTC

End: 2024-09-30 23:59 UTC

In [27]:
# ---- CONFIG: western NC county FIPS (3-digit strings) ----
WESTERN_NC_COUNTYFPS = {
    '001','005','009','011','021','027','039','043','045','087','089',
    '099','113','115','121','149','161','175','189','199','221','193','197'
}
# ----------------------------------------------------------

# parse datetime (UTC)
combined_df['observation_datetime'] = pd.to_datetime(combined_df['observation_datetime'], utc=True, errors='coerce')

# auto-detect station id column in combined_df (common names)
possible_keys = ['station_id','station','nearest_station','station_code','id']
left_key = next((c for c in possible_keys if c in combined_df.columns), None)
right_key = 'nearest_station' 

if left_key is None:
    raise RuntimeError(f"Could not find a station id column in combined_df. Expected one of {possible_keys}.")

# merge TRACT onto combined_df
merged = combined_df.merge(tract_station_map[['TRACT', right_key]], left_on=left_key, right_on=right_key, how='left')

# extract county FIPS from TRACT (state(2)+county(3)+tract(6))
merged['TRACT'] = merged['TRACT'].astype(str)
merged['countyfp'] = merged['TRACT'].str[2:5]

# helene window (UTC)
start_helene = pd.Timestamp("2024-09-26 00:00", tz="UTC")
end_helene   = pd.Timestamp("2024-09-30 23:59", tz="UTC")

# flags
merged['in_helene_window'] = merged['observation_datetime'].between(start_helene, end_helene)
merged['is_western_nc'] = merged['countyfp'].isin(WESTERN_NC_COUNTYFPS)
merged['drop_for_helene'] = merged['in_helene_window'] & merged['is_western_nc']

# diagnostics
print("Total rows:", len(merged))
print("In Helene window (anywhere):", int(merged['in_helene_window'].sum()))
print("In Helene window AND western NC (to drop):", int(merged['drop_for_helene'].sum()))

Total rows: 39138145
In Helene window (anywhere): 311146
In Helene window AND western NC (to drop): 33699
Rows after filtering: 39104446


In [28]:
combined_df_filtered = merged

# prepare daily station weather (station x date)
combined_df_filtered['observation_datetime'] = pd.to_datetime(combined_df_filtered['observation_datetime'], utc=True)
combined_df_filtered['date'] = combined_df_filtered['observation_datetime'].dt.date

weather_daily = combined_df_filtered.groupby(['station','date'], as_index=False).agg(
    mean_wind = ('windspeed10m_mph', 'mean'),
    max_gust  = ('gustspeed10m_mph', 'max'),
    total_precip = ('precip_in', 'sum'),
    mean_soilmoist = ('soilmoist20cm', 'mean')
)

# ensure TRACT and station mapping present on outages
outage_events['TRACT'] = outage_events['TRACT'].astype(str)
tract_station_map['TRACT'] = tract_station_map['TRACT'].astype(str)

if 'nearest_station' not in outage_events.columns:
    outage_events = outage_events.merge(
        tract_station_map[['TRACT','nearest_station']],
        on='TRACT', how='left'
    )

# make outage-days (one row per outage-day)
outage_events['start_date'] = pd.to_datetime(outage_events['outage_start_estimate'], utc=True).dt.date
outage_events['end_date']   = pd.to_datetime(outage_events['outage_end_estimate'], utc=True).dt.date

outage_events['outage_dates'] = outage_events.apply(
    lambda r: pd.date_range(r['start_date'], r['end_date'], freq='D').date, axis=1
)

outage_days = outage_events[['outage_identifer','TRACT','nearest_station','outage_dates']].explode('outage_dates')
outage_days = outage_days.rename(columns={'outage_dates':'date'}).dropna(subset=['nearest_station'])

# join outage-days to station daily weather
outage_weather_daily = outage_days.merge(
    weather_daily,
    left_on=['nearest_station','date'],
    right_on=['station','date'],
    how='left'
)

# aggregate to one row per TRACT
tract_weather_summary = outage_weather_daily.groupby('TRACT', as_index=False).agg(
    max_gust_during_outage_days = ('max_gust', 'max'),
    mean_wind_on_outage_days    = ('mean_wind', 'mean'),
    total_precip_on_outage_days = ('total_precip', 'sum'),
    mean_soilmoist_on_outage_days = ('mean_soilmoist', 'mean'),
    outage_weather_day_count = ('date', lambda x: x.nunique())
)

In [29]:
tract_weather = tract_weather_summary

# Save dataframes to csvs for later use

In [31]:
tract_socioeconomic.to_csv('data_processed/PCA_data/tract_socioeconomic.csv', index=False)
tract_socioeconomic_unscaled.to_csv('data_processed/PCA_data/tract_socioeconomic_unscaled.csv', index=False)
outage_events.to_csv('data_processed/PCA_data/outage_events.csv', index=False)
tract_outages.to_csv('data_processed/PCA_data/tract_outages.csv', index=False)
tract_weather.to_csv('data_processed/PCA_data/tract_weather.csv', index=False)